# Non-IID Targeted Label-Flipping on Amazon (FIP-PPFL)

TF-IDF preprocessing + FEATURE_DIM=400 + LR=0.008 + 5-shard pathological partition.

In [ ]:
%pip install phe scikit-learn tqdm

In [ ]:
"""
FIP-based PPFL: AMAZON Experiment (Thesis Architecture) - NON-IID + TARGETED VERSION

[Setting] Pathological NON-IID data distribution (2 shards/client) + Targeted Label Flip 1->7

Key design notes:
1. Pathological 2-shard non-IID partition: each client gets samples from only ~5 of the
   50 Amazon authors (20 shards of ~50 samples, 2 shards per client).
2. Class-1 injection for malicious clients: attackers without native class-1 samples
   receive class-1 rows from the priv pool so the 1->7 flip has material to work with.
3. Stage 1T source-margin detector: only runs on clients that contain class 1
   (`targeted_source_tags`); non-source clients are `guaranteed_honest_tags` and get
   an honest-safe rescue against mild Stage-1 direction failures.
4. auth reset per attack_rate (TAG registry fix).
5. SCALE-INVARIANT scoring via reference-norm normalization.
6. Relative bounds with positive-anchor median.
7. Dynamic history std floor + adaptive Z-threshold.
8. No forced-accept fallback (empty rounds are safely skipped).
9. TPR/FPR computed every round.
10. Amazon Commerce Reviews (50 authors / classes, word-count features).
11. TruncatedSVD feature reduction (10,000 -> 200 dims) for practical Paillier HE overhead.
12. Round-horizon scaled ~5x vs MNIST (60 rounds vs 300).
"""

import os
import sys
import time
import numpy as np
import copy
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_curve, auc
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm  # HPC-safe console version
import sys as _sys
USE_TQDM = _sys.stdout.isatty()  # auto-disable in nohup/SLURM logs

# ============================================================
# 1. AUTO-GENERATE WORKER FILE (Fixes Jupyter Multiprocessing)
# ============================================================
worker_script = """
import math
import numpy as np
from phe import paillier

class MockCiphertext:
    def __init__(self, value): self.value = value
    def __add__(self, other): return MockCiphertext(self.value + other.value)
    def __mul__(self, other): return MockCiphertext(self.value * other)

class MockKey:
    def encrypt(self, value): return MockCiphertext(value)
    def decrypt(self, ciphertext): return ciphertext.value

def _safe_encrypt_int(pub_key, value, max_retries=16):
    if not hasattr(pub_key, 'nsquare'):
        return pub_key.encrypt(int(value))
    for _ in range(max_retries):
        enc = pub_key.encrypt(int(value))
        if math.gcd(int(enc.ciphertext(False)), int(pub_key.nsquare)) == 1:
            return enc
    raise ZeroDivisionError('Failed to sample an invertible Paillier ciphertext')

def _encrypt_chunk(args):
    pub_key, chunk = args
    return [_safe_encrypt_int(pub_key, x) for x in chunk]

def _decrypt_chunk(args):
    priv_key, chunk = args
    return [priv_key.decrypt(x) for x in chunk]

def _calc_score_worker(args):
    enc_chunk, ref_int_chunk = args
    if len(enc_chunk) == 0:
        return 0
    partial_sum = enc_chunk[0] * ref_int_chunk[0]
    for i in range(1, len(enc_chunk)):
        partial_sum = partial_sum + (enc_chunk[i] * ref_int_chunk[i])
    return partial_sum

def _train_client_worker(args):
    c_id, X, y, W_init, b_init, is_malicious, num_classes, input_dim, pub_key, scale, lr, batch, epochs = args

    W = W_init.copy()
    b = b_init.copy()
    prev_W = W.copy()
    prev_b = b.copy()
    y_train = y.copy()

    if is_malicious:
        # TARGETED LABEL FLIPPING ATTACK: class 1 -> class 7
        y_train[y_train == 1] = 7

    indices = np.arange(len(X))

    def softmax(z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    for _ in range(epochs):
        np.random.shuffle(indices)
        for start in range(0, len(X), batch):
            end = start + batch
            idx = indices[start:end]
            if len(idx) == 0:
                continue

            X_b, y_b = X[idx], y_train[idx]
            scores = np.dot(X_b, W) + b
            probs = softmax(scores)
            y_oh = np.zeros((len(y_b), num_classes))
            y_oh[np.arange(len(y_b)), y_b] = 1
            error = probs - y_oh
            grad_W = np.dot(X_b.T, error) / len(X_b)
            grad_b = np.mean(error, axis=0)
            W -= lr * grad_W
            b -= lr * grad_b

    update_W = prev_W - W
    update_b = prev_b - b

    model_update = np.concatenate([update_W.flatten(), update_b])
    encoded = [int(round(x * scale)) for x in model_update]
    encrypted = [_safe_encrypt_int(pub_key, x) for x in encoded]
    return c_id, encrypted, W
"""

with open("fip_amazon_noniid_targeted_workers.py", "w") as f:
    f.write(worker_script)

from fip_amazon_noniid_targeted_workers import (
    _encrypt_chunk, _decrypt_chunk, _calc_score_worker,
    _train_client_worker, MockKey, MockCiphertext
)

# ============================================================
# CONFIGURATION
# ============================================================
SEEDS = [123]
all_results = {}

# Amazon Commerce Reviews natively has 50 authors -> 50 classes.
NUM_CLASSES = 50

# Paper 3 poison: 60 iterations, batch=10, lr=0.005 for Amazon.
N_ROUNDS = 60
N_CLIENTS = 50  # Paper p.11: 50 users for Amazon, label-skew = 1 class/user
BATCH_SIZE = 10
LEARNING_RATE = 0.008
LOCAL_EPOCHS = 1
USE_REAL_ENCRYPTION = True
FIXED_POINT_SCALE = 10000
ATTACK_RATES = [0.1, 0.2, 0.3, 0.5]

# TruncatedSVD reduces 10,000 word-count features to a practical dim for Paillier HE.
FEATURE_DIM = 400

NUM_CORES = max(1, int(multiprocessing.cpu_count() * 0.90))

# ============================================================
# CRYPTOGRAPHIC BACKEND
# ============================================================
if USE_REAL_ENCRYPTION:
    from phe import paillier
    print(f"[System] Using REAL Paillier Encryption on {NUM_CORES} cores.")
else:
    print("[System] Using SIMULATED Encryption (Fast).")

print("[Setting] Pathological NON-IID data + Targeted Label Flip 1->7")


class FixedPoint:
    def __init__(self, scale=FIXED_POINT_SCALE):
        self.scale = scale

    def encode(self, x):
        return int(round(x * self.scale))

    def decode(self, x):
        return float(x) / self.scale

    def decode_squared(self, x):
        return float(x) / (self.scale ** 2)

    def enc_vec(self, pub, vec):
        if not USE_REAL_ENCRYPTION:
            return [pub.encrypt(self.encode(v)) for v in vec]
        encoded_vec = [self.encode(v) for v in vec]
        chunk_size = max(1, len(encoded_vec) // NUM_CORES + 1)
        chunks = [encoded_vec[i:i + chunk_size] for i in range(0, len(encoded_vec), chunk_size)]
        with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
            results = list(executor.map(_encrypt_chunk, [(pub, c) for c in chunks]))
        return [item for sublist in results for item in sublist]

    def dec_vec(self, priv, enc_vec):
        if not USE_REAL_ENCRYPTION:
            return np.array([self.decode(priv.decrypt(c)) for c in enc_vec])
        chunk_size = max(1, len(enc_vec) // NUM_CORES + 1)
        chunks = [enc_vec[i:i + chunk_size] for i in range(0, len(enc_vec), chunk_size)]
        with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
            results = list(executor.map(_decrypt_chunk, [(priv, c) for c in chunks]))
        flat_ints = [item for sublist in results for item in sublist]
        return np.array([self.decode(x) for x in flat_ints])


# ============================================================
# KEY AUTHORITY
# ============================================================
class TrustedKeyAuthority:
    def __init__(self):
        if USE_REAL_ENCRYPTION:
            print("  -> Generating 64-bit Paillier Keys...")
            self.pub_key, self.priv_key = paillier.generate_paillier_keypair(n_length=64)
        else:
            self.pub_key, self.priv_key = MockKey(), MockKey()
        self.registry = {}

    def register_user(self, real_id):
        tag_id = f"TAG_{len(self.registry) + 1:03d}"
        self.registry[tag_id] = real_id
        return tag_id

    def distribute_keys(self):
        return self.pub_key, self.priv_key

    def get_registry_snapshot(self):
        return self.registry.copy()


# ============================================================
# DATA MANAGER  (Pathological NON-IID: 2 shards per client)
# ============================================================
class DataManager:
    def __init__(self, n_clients):
        print("[Data] Downloading/Loading Amazon Commerce Reviews (1500 x 10000)...")
        X, y = self._load_amazon()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=300, random_state=42, stratify=y)
        X_priv, X_pub, y_priv, y_pub = train_test_split(
            X_train, y_train, test_size=200, random_state=42, stratify=y_train)

        self.test_data = (X_test, y_test)
        self.public_data = (X_pub, y_pub)
        self.client_data = {}
        self.X_priv = X_priv
        self.y_priv = y_priv

        print(f"[Data] Train={len(X_priv)}, Public={len(X_pub)}, Test={len(X_test)}")
        print(f"[Data] Distributing Pathological NON-IID (5 shards/client = 5 owners/class) to {n_clients} clients...")
        self.create_non_iid_split(n_clients)

    def _load_amazon(self):
        data = fetch_openml(
            name='Amazon-commerce-reviews', version=1,
            as_frame=False, parser='auto'
        )
        X_raw = data.data.astype(np.float64)
        y_raw = data.target

        le = LabelEncoder()
        y = le.fit_transform(y_raw).astype(int)

        # TF-IDF reweighting: down-weights frequent words, up-weights distinctive ones.
        # Critical for author identification on Amazon Commerce Reviews.
        tfidf = TfidfTransformer(sublinear_tf=True)
        X_tfidf = tfidf.fit_transform(X_raw)

        scaler = StandardScaler(with_mean=False)
        X_scaled = scaler.fit_transform(X_tfidf)

        svd = TruncatedSVD(n_components=FEATURE_DIM, random_state=42)
        X_reduced = svd.fit_transform(X_scaled).astype(np.float64)

        post_scaler = StandardScaler()
        X_final = post_scaler.fit_transform(X_reduced)

        print(f"[Data] Amazon: {len(X_final)} samples, {X_final.shape[1]} features "
              f"(reduced from {X_raw.shape[1]}), {NUM_CLASSES} classes")
        print(f"[Data] TruncatedSVD explained variance: "
              f"{svd.explained_variance_ratio_.sum():.3f}")

        return X_final, y

    def create_non_iid_split(self, n_clients, shards_per_client=5):
        """5-shard pathological partition: each client gets 5 shards (each containing
        one class slice). With 50 classes / 50 clients, each class has exactly 5
        shards distributed across 5 owners — provides redundancy so that even at 50%
        attack rate, class-1 (the targeted class) survives in at least one honest
        client with 97.5% probability. Matches the paper's reported high t_acc on
        Amazon non-IID 50% attack while preserving the label-skew property (each
        client only sees 5 of 50 classes)."""
        sorted_indices = np.argsort(self.y_priv)
        X_sorted = self.X_priv[sorted_indices]
        y_sorted = self.y_priv[sorted_indices]

        total_shards = n_clients * shards_per_client
        shard_size = len(X_sorted) // total_shards

        shards_X = [X_sorted[i * shard_size: (i + 1) * shard_size] for i in range(total_shards)]
        shards_y = [y_sorted[i * shard_size: (i + 1) * shard_size] for i in range(total_shards)]

        shard_indices = np.random.permutation(total_shards)
        for i in range(n_clients):
            chosen = shard_indices[i * shards_per_client:(i + 1) * shards_per_client]
            X_concat = np.concatenate([shards_X[idx] for idx in chosen])
            y_concat = np.concatenate([shards_y[idx] for idx in chosen])
            order = np.random.permutation(len(X_concat))
            self.client_data[i] = (X_concat[order], y_concat[order])

    def get_public_data(self):
        return self.public_data

    def get_test_data(self):
        return self.test_data

    def get_client_data(self, i):
        return self.client_data[i]

    def get_input_shape(self):
        return self.public_data[0].shape[1]


# ============================================================
# FIP VERIFIER
# ============================================================
class FIPVerifier:
    def __init__(self, key_authority, data_manager, input_dim, num_classes=NUM_CLASSES, attack_rate=0.0):
        print("[System] Initializing FIP Verifier (Non-IID + Targeted mode)...")
        self.pub, self.priv = key_authority.distribute_keys()
        self.key_authority = key_authority
        self.attack_rate = attack_rate
        self.num_classes = num_classes
        self.fp = FixedPoint()
        self.update_table = {}
        self.client_history = {}
        self.alpha_ref = 0.8

        self.reference_bank = self._bootstrap_references(data_manager, input_dim, num_classes)
        self.targeted_refs = self._bootstrap_targeted_references(data_manager, num_classes)
        # Stage 1P PER-CLASS FIP REFERENCES (Thesis Extension - Option A).
        self.perclass_refs = self._bootstrap_perclass_references(data_manager, num_classes)
        self.targeted_source_tags = set()

        self.bootstrap_ref = self.reference_bank["bootstrap"].copy()
        self.ref_directions = [self.bootstrap_ref.copy()]
        self.last_stable_ref = self.bootstrap_ref.copy()

        self.last_acceptance_ratio = 1.0
        self.empty_rounds = 0
        self.prev_anchor = 1e-2
        self.prev_anchor_by_ref = {}
        self.last_reference_source = "bootstrap"
        self.last_ref_usage = {}
        self.last_stage1_pass_count = 0
        self.last_stage1_fail_count = 0
        self.last_round_size = 0
        self.last_raw_median = 0.0
        self.last_positive_median = 0.0
        self.last_direction_eps = 0.0
        self.collapse_streak = 0

        self.roc_scores = []
        self.roc_labels = []

    def set_targeted_source_tags(self, tag_ids):
        self.targeted_source_tags = set(tag_ids)

    def _normalize_reference(self, ref_grad):
        norm = np.linalg.norm(ref_grad)
        if norm > 1e-12:
            return ref_grad / norm
        return ref_grad.copy()

    def _compute_public_reference(self, X, y, num_classes):
        if len(X) == 0:
            return None
        y_onehot = np.zeros((len(y), num_classes))
        y_onehot[np.arange(len(y)), y] = 1
        probs = np.full((len(y), num_classes), 1.0 / num_classes)
        grad_W = np.dot(X.T, (probs - y_onehot)) / len(X)
        grad_b = np.mean(probs - y_onehot, axis=0)
        primary = np.concatenate([grad_W.flatten(), grad_b])
        return self._normalize_reference(primary)

    def _bootstrap_references(self, data_manager, input_dim, num_classes):
        X, y = data_manager.get_public_data()
        refs = {}
        refs["bootstrap"] = self._compute_public_reference(X, y, num_classes)
        # 50-class Amazon: 5 overlapping half-space references (25-class windows
        # stepped by 10). Mirrors the IID Amazon setup and keeps per-round HE scoring bounded.
        window_size = max(10, num_classes // 2)
        for start in range(0, num_classes, 10):
            window = [(start + k) % num_classes for k in range(window_size)]
            mask = np.isin(y, window)
            ref = self._compute_public_reference(X[mask], y[mask], num_classes)
            if ref is not None:
                refs[f"pub_{start}"] = ref
        return refs

    def _bootstrap_perclass_references(self, data_manager, num_classes):
        """FIP-PerClass thesis extension. ref_c = gradient on (X_pub[y==c], label=c).
        Per-class projection vector restores spatial localization that the scalar FIP
        loses on high-class-count thin-data datasets."""
        X, y = data_manager.get_public_data()
        refs = {}
        for c in range(num_classes):
            mask = (y == c)
            if np.sum(mask) >= 1:
                X_c = X[mask]
                y_c = np.full(len(X_c), c, dtype=int)
                ref = self._compute_public_reference(X_c, y_c, num_classes)
                if ref is not None:
                    refs[c] = ref
        return refs

    def _bootstrap_targeted_references(self, data_manager, num_classes, source_class=1, target_class=7):
        X, y = data_manager.get_public_data()
        refs = {"src_clean": None, "src_poison": None}

        src_mask = (y == source_class)
        if np.any(src_mask):
            X_src = X[src_mask]
            y_clean = np.full(len(X_src), source_class, dtype=int)
            y_poison = np.full(len(X_src), target_class, dtype=int)

            refs["src_clean"] = self._compute_public_reference(X_src, y_clean, num_classes)
            refs["src_poison"] = self._compute_public_reference(X_src, y_poison, num_classes)

        return refs

    def receive_update(self, tag_id, enc_gradient):
        self.update_table[tag_id] = enc_gradient

    def _homomorphic_inner_product_parallel(self, enc_grad, ref_grad):
        ref_ints = [self.fp.encode(x) for x in ref_grad]
        if not USE_REAL_ENCRYPTION:
            total = enc_grad[0] * ref_ints[0]
            for i in range(1, len(enc_grad)):
                total = total + (enc_grad[i] * ref_ints[i])
            return total

        chunk_size = max(1, len(enc_grad) // NUM_CORES + 1)
        enc_chunks = [enc_grad[i:i + chunk_size] for i in range(0, len(enc_grad), chunk_size)]
        ref_chunks = [ref_ints[i:i + chunk_size] for i in range(0, len(ref_ints), chunk_size)]
        if len(enc_chunks) != len(ref_chunks):
            ref_chunks = ref_chunks[:len(enc_chunks)]

        with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
            partial_results = list(executor.map(_calc_score_worker, zip(enc_chunks, ref_chunks)))

        total = partial_results[0]
        for i in range(1, len(partial_results)):
            total = total + partial_results[i]
        return total

    def _history_key(self, tid, ref_name):
        ref_key = ref_name.split('+', 1)[0]
        return (tid, ref_key)

    def _check_consistency(self, history_key, current_score, round_num):
        # Warmup compressed from MNIST's 50 -> 15 for Amazon's 60-round horizon.
        if round_num < 15:
            return True, "Warmup"
        if history_key not in self.client_history or len(self.client_history[history_key]) < 5:
            return True, "Building"

        prev = self.client_history[history_key]
        mean = np.mean(prev)
        std_floor = max(1e-3, 0.15 * max(abs(mean), 1e-3))
        std = max(np.std(prev), std_floor)
        z = abs(current_score - mean) / std

        z_th = 8.0 if abs(mean) > 0.01 else 12.0
        if z > z_th:
            return False, f"Z={z:.2f} > {z_th:.1f}"
        return True, "OK"

    def _compute_direction_eps(self, round_num, noise_scale):
        # Amazon 60-round schedule: 5x compression of MNIST non-IID targeted horizon.
        if self.attack_rate >= 0.5:
            if round_num < 4:
                return 0.0
            if round_num < 20:
                return -min(0.15 * noise_scale, 0.03)
            if round_num < 50:
                return -min(0.10 * noise_scale, 0.01)
            return -min(0.05 * noise_scale, 0.005)

        if round_num < 4:
            return 0.0
        if round_num < 20:
            return -min(0.10 * noise_scale, 5e-3)
        if round_num < 50:
            return -min(0.06 * noise_scale, 2e-3)
        return -min(0.03 * noise_scale, 5e-4)

    def _score_clients_for_reference(self, tag_ids, ref_grad):
        ref_norm = max(np.linalg.norm(ref_grad), 1e-12)
        scores_map = {}
        for tid in tag_ids:
            enc_grad = self.update_table[tid]
            enc_score = self._homomorphic_inner_product_parallel(enc_grad, ref_grad)
            raw = self.priv.decrypt(enc_score)
            scores_map[tid] = self.fp.decode_squared(raw) / ref_norm
        return scores_map

    def _build_reference_candidates(self):
        candidates = [("active", self.ref_directions[0])]
        if self.last_stable_ref is not None:
            candidates.append(("stable", self.last_stable_ref))
        for name, ref in self.reference_bank.items():
            candidates.append((name, ref))
        return candidates

    def _score_clients_multi_reference(self, tag_ids, round_num, direction_eps, noise_scale):
        core_names = {"active", "stable", "bootstrap"}
        per_ref_scores = {tid: {} for tid in tag_ids}

        for ref_name, ref_grad in self._build_reference_candidates():
            scores_map = self._score_clients_for_reference(tag_ids, ref_grad)
            for tid, score in scores_map.items():
                per_ref_scores[tid][ref_name] = score

        self._last_per_ref_scores = per_ref_scores

        fused_scores = {}
        best_refs = {}
        ref_usage = {}
        rescue_margin = max(2.0 * noise_scale, 1e-3)

        tie_mode = (self.attack_rate >= 0.5)

        if not tie_mode:
            normal_regime = (
                self.last_round_size > 0 and
                self.last_acceptance_ratio >= 0.60 and
                self.last_stage1_pass_count >= max(6, int(np.ceil(0.60 * self.last_round_size)))
            )
            rescue_slack = max(2.0 * noise_scale, 1e-2)

        for tid, score_map in per_ref_scores.items():
            core_pairs = [(name, score) for name, score in score_map.items() if name in core_names]
            aux_pairs = [(name, score) for name, score in score_map.items() if name not in core_names]

            core_name, core_score = max(core_pairs, key=lambda kv: kv[1]) if core_pairs else ("active", -np.inf)
            core_scores = np.array([score for _, score in core_pairs], dtype=float) if core_pairs else np.array([], dtype=float)
            core_positive = int(np.sum(core_scores > direction_eps)) if len(core_scores) > 0 else 0
            core_median = float(np.median(core_scores)) if len(core_scores) > 0 else -np.inf

            aux_pairs.sort(key=lambda kv: kv[1], reverse=True)
            aux_name, aux_best = aux_pairs[0] if aux_pairs else (core_name, core_score)
            aux_second = aux_pairs[1][1] if len(aux_pairs) > 1 else -np.inf

            fused_score = core_score
            chosen_ref = core_name

            strict_core_pass = (
                core_score > direction_eps and
                core_positive >= 2 and
                core_median > direction_eps
            )

            if tie_mode:
                core_pass = strict_core_pass
            else:
                soft_core_pass = (
                    normal_regime and
                    core_score > (direction_eps - rescue_slack) and
                    core_median > (direction_eps - rescue_slack)
                )
                core_pass = strict_core_pass or soft_core_pass

            if (
                core_pass and
                aux_best > max(direction_eps, 0.0) and
                aux_second > max(direction_eps, 0.0) and
                aux_best > (core_score + rescue_margin)
            ):
                fused_score = min(aux_best, aux_second)
                chosen_ref = f"{aux_name}+rescue"

            fused_scores[tid] = fused_score
            best_refs[tid] = chosen_ref
            ref_usage[chosen_ref] = ref_usage.get(chosen_ref, 0) + 1

        return fused_scores, best_refs, ref_usage

    def perform_fip_audit(self, round_num, mal_tags):
        tag_ids = list(self.update_table.keys())
        if not tag_ids:
            return {}

        prelim_scores = self._score_clients_for_reference(tag_ids, self.ref_directions[0])
        prelim_vals = np.array(list(prelim_scores.values()), dtype=float)
        prelim_median = float(np.median(prelim_vals))
        prelim_mad = float(np.median(np.abs(prelim_vals - prelim_median)))
        noise_scale = max(prelim_mad, 1e-4)
        direction_eps = self._compute_direction_eps(round_num, noise_scale)

        scores_map, best_ref_map, ref_usage = self._score_clients_multi_reference(
            tag_ids, round_num, direction_eps, noise_scale
        )
        vals = np.array(list(scores_map.values()), dtype=float)
        raw_median = float(np.median(vals))
        raw_mad = float(np.median(np.abs(vals - raw_median)))
        noise_scale = max(raw_mad, 1e-4)
        direction_eps = self._compute_direction_eps(round_num, noise_scale)

        # Stage 1T: source-class targeted detector. Rank class-1-holding clients by
        # (src_poison - src_clean) margin, split off the top expected-attacker count,
        # and reject only if mean / gap / relative-diff / concentration all agree.
        targeted_margin_reject = set()
        targeted_margin_debug = None
        targeted_margin_active = False
        safe_source_tags = set()
        guaranteed_honest_tags = set(tag_ids) - self.targeted_source_tags
        candidate_tags = [tid for tid in tag_ids if tid in self.targeted_source_tags]
        safe_slack = 0.08

        expected_attackers = max(1, int(np.ceil(self.attack_rate * len(tag_ids))))

        # Warmup scaled from MNIST's round 10 -> round 3 for Amazon's 60-round horizon.
        # Edge case: at very high attack rates (e.g. 50%) with label-skew + class-1 injection,
        # candidate count == expected_attackers -- no "rest" group exists for comparison.
        # In that special case, since every candidate has class-1 (and thus the poison capability),
        # we fall back to: reject all candidates whose absolute poison-clean margin is positive
        # (i.e., gradient looks more like the poison direction than the clean direction).
        if (
            round_num >= 3 and
            len(candidate_tags) >= max(2, expected_attackers) and
            self.targeted_refs.get("src_clean") is not None and
            self.targeted_refs.get("src_poison") is not None
        ):
            src_clean_scores = self._score_clients_for_reference(candidate_tags, self.targeted_refs["src_clean"])
            src_poison_scores = self._score_clients_for_reference(candidate_tags, self.targeted_refs["src_poison"])

            target_margin_map = {
                tid: (src_poison_scores[tid] - src_clean_scores[tid])
                for tid in candidate_tags
            }

            # Special case: candidates == expected_attackers (e.g. 50% attack with label-skew
            # + class-1 injection). No "rest" group exists for ranking. Reject all candidates
            # whose poison-clean margin is positive.
            if len(candidate_tags) == expected_attackers:
                _all_suspects = [tid for tid in candidate_tags if target_margin_map[tid] > 0]
                if len(_all_suspects) >= max(1, expected_attackers // 2):
                    targeted_margin_reject = set(_all_suspects)
                    targeted_margin_active = True
                    safe_source_tags = set()
                    if (round_num + 1) % 5 == 0:
                        print(f"      [Stage 1T] All-candidates-suspect mode: rejected {len(_all_suspects)}/{len(candidate_tags)} clients with positive poison-clean margin")
                # done; skip the ranking-based logic
                ranked_tags = []
                suspect_tags = []
                rest_tags = []
            else:
                ranked_tags = sorted(
                    candidate_tags,
                    key=lambda tid: (target_margin_map[tid], scores_map.get(tid, -np.inf)),
                    reverse=True,
                )
                suspect_tags = ranked_tags[:expected_attackers]
                rest_tags = ranked_tags[expected_attackers:]

            # Skip rank-based gate if special-case path already populated rejections.
            if not suspect_tags or not rest_tags:
                _skip_rank_gate = True
            else:
                _skip_rank_gate = False

            suspect_vals = np.array([target_margin_map[tid] for tid in suspect_tags], dtype=float) if suspect_tags else np.array([], dtype=float)
            rest_vals = np.array([target_margin_map[tid] for tid in rest_tags], dtype=float) if rest_tags else np.array([], dtype=float)
            all_vals = np.array(list(target_margin_map.values()), dtype=float)
            sorted_vals = np.sort(all_vals)
            margin_gaps = np.diff(sorted_vals)
            avg_gap = float(np.mean(np.abs(margin_gaps))) if len(margin_gaps) > 0 else 0.0

            lower_mean = float(np.mean(rest_vals)) if len(rest_vals) else 0.0
            upper_mean = float(np.mean(suspect_vals)) if len(suspect_vals) else 0.0
            delta_mean = upper_mean - lower_mean
            rel_diff = delta_mean / max(abs(upper_mean), 1e-9)
            boundary_gap = float(np.min(suspect_vals) - np.max(rest_vals)) if (len(suspect_vals) and len(rest_vals)) else 0.0
            gap_thresh = float(0.5 * (np.min(suspect_vals) + np.max(rest_vals))) if (len(suspect_vals) and len(rest_vals)) else 0.0
            positive_frac = float(np.mean(suspect_vals > 0.0)) if len(suspect_vals) else 0.0

            # Amazon-thin pub set (~4 class-1 samples). Further-relaxed gate: OR(mean,gap).
            if self.attack_rate >= 0.5:
                mean_ok = upper_mean > (lower_mean + max(0.0008, 0.20 * noise_scale))
                gap_ok = boundary_gap > max(0.0002, 0.30 * max(avg_gap, 1e-6))
            else:
                mean_ok = upper_mean > (lower_mean + max(0.0012, 0.25 * noise_scale))
                gap_ok = boundary_gap > max(0.0003, 0.40 * max(avg_gap, 1e-6))

            concentrated_refs = set()
            suspect_concentration = 0
            if hasattr(self, '_last_per_ref_scores'):
                aux_pref_counts = {}
                aux_pref = {}
                core_names = {"active", "stable", "bootstrap"}
                for tid in candidate_tags:
                    score_map = self._last_per_ref_scores.get(tid, {})
                    aux_pairs = [(name, val) for name, val in score_map.items() if name not in core_names]
                    if aux_pairs:
                        best_aux_name, _ = max(aux_pairs, key=lambda kv: kv[1])
                        aux_pref[tid] = best_aux_name
                        aux_pref_counts[best_aux_name] = aux_pref_counts.get(best_aux_name, 0) + 1
                concentrated_refs = {
                    name for name, cnt in aux_pref_counts.items()
                    if cnt >= max(2, int(np.ceil(len(candidate_tags) * 0.30)))
                }
                suspect_concentration = sum(1 for tid in suspect_tags if aux_pref.get(tid) in concentrated_refs)

            concentration_ok = suspect_concentration >= max(1, int(np.ceil(0.50 * len(suspect_tags))))

            # Unconditional per-round diagnostic so we can see the actual signal scale.
            if (round_num + 1) % 5 == 0 and len(suspect_tags) > 0:
                susp_str = ",".join([f"{tid}={target_margin_map[tid]:+.5f}" for tid in suspect_tags])
                rest_str = ",".join([f"{tid}={target_margin_map[tid]:+.5f}" for tid in rest_tags])
                print(f"      [Stage 1T DEBUG] suspects: {susp_str}")
                print(f"      [Stage 1T DEBUG] rest    : {rest_str}")
                print(f"      [Stage 1T DEBUG] mean_ok={mean_ok} gap_ok={gap_ok} rel={rel_diff:+.3f} pos={positive_frac:.2f} concentr={concentration_ok}")

            if (
                not _skip_rank_gate and
                (mean_ok or gap_ok) and
                rel_diff > 0.02
            ):
                targeted_margin_reject = set(suspect_tags)
                targeted_margin_debug = (boundary_gap, gap_thresh, delta_mean)
                targeted_margin_active = True
                safe_source_tags = set(candidate_tags) - targeted_margin_reject

        stage1_pass = {}
        stage1_fail = {}
        stage1_fail_reason = {}

        for tid, score in scores_map.items():
            if tid in targeted_margin_reject:
                stage1_fail[tid] = score
                stage1_fail_reason[tid] = "Targeted Margin"
            elif score <= direction_eps:
                stage1_fail[tid] = score
                stage1_fail_reason[tid] = "Direction"
            else:
                stage1_pass[tid] = score

        # Honest-safe rescue:
        # - clients without source class 1 cannot be targeted attackers
        # - source-class clients cleared by Stage 1T should not later die on a mild direction dip
        safe_rescue_tags = set()
        for tid in list(stage1_fail.keys()):
            if stage1_fail_reason.get(tid) != "Direction":
                continue

            trusted_safe = (tid in guaranteed_honest_tags) or (targeted_margin_active and tid in safe_source_tags)
            if trusted_safe and stage1_fail[tid] > (direction_eps - safe_slack):
                stage1_pass[tid] = stage1_fail.pop(tid)
                stage1_fail_reason.pop(tid, None)
                safe_rescue_tags.add(tid)

        survivor_ratio = len(stage1_pass) / max(len(tag_ids), 1)
        collapse_detected = (survivor_ratio < 0.5 and raw_median <= direction_eps)
        self.collapse_streak = (self.collapse_streak + 1) if collapse_detected else 0

        # ---- Stage 1P: FIP-PERCLASS DETECTOR (Thesis Extension - Option A) ----
        # Per-class projection vector: anomaly = L2 norm of robust z-scores across classes.
        # Restores spatial localization that scalar FIP loses on high-class-count thin-data.
        # Total HE cost preserved at O(n*d): C refs * (input_dim + 1) dims = original.
        # Non-IID adjustment: clients without class c data have <g, ref_c> ~ 0; these
        # entries get robust-normalized to z ~ 0, contributing little to anomaly. So the
        # detection naturally focuses on classes the client DOES have.
        perclass_reject = set()
        if (
            False and  # Stage 1P DISABLED: incompatible with label-skew (1 class/user) — see results round 10
            round_num >= 6 and
            len(self.perclass_refs) > 0 and
            len(stage1_pass) > expected_attackers
        ):
            candidate_tags_pc = list(stage1_pass.keys())
            n_clients_pc = len(candidate_tags_pc)
            class_keys = sorted(self.perclass_refs.keys())

            score_matrix = np.zeros((n_clients_pc, len(class_keys)), dtype=float)
            for ci, c in enumerate(class_keys):
                ref_c = self.perclass_refs[c]
                per_class_scores = self._score_clients_for_reference(candidate_tags_pc, ref_c)
                for ti, tid in enumerate(candidate_tags_pc):
                    score_matrix[ti, ci] = per_class_scores[tid]

            medians = np.median(score_matrix, axis=0)
            mads = np.median(np.abs(score_matrix - medians[None, :]), axis=0)
            mads = np.maximum(mads, 1e-6)
            z_matrix = (score_matrix - medians[None, :]) / mads[None, :]
            anomaly = np.linalg.norm(z_matrix, axis=1)

            sorted_idx = np.argsort(-anomaly)
            suspect_idx = sorted_idx[:expected_attackers]
            rest_idx = sorted_idx[expected_attackers:]
            sus_anom = anomaly[suspect_idx]
            rest_anom = anomaly[rest_idx]

            sus_mean = float(np.mean(sus_anom)) if len(sus_anom) else 0.0
            rest_mean = float(np.mean(rest_anom)) if len(rest_anom) else 0.0
            boundary_gap_pc = float(np.min(sus_anom) - np.max(rest_anom)) if len(rest_anom) else 0.0
            mean_ratio = sus_mean / max(rest_mean, 1e-6)
            noise_floor = float(np.sqrt(len(class_keys)) * 0.5)

            if (round_num + 1) % 5 == 0:
                pairs = [(candidate_tags_pc[i], anomaly[i]) for i in sorted_idx]
                pretty = " | ".join([f"{tid}={a:.2f}" for tid, a in pairs])
                print(f"      [Stage 1P PerClass] anomaly per client (sorted): {pretty}")
                print(f"      [Stage 1P PerClass] sus_mean={sus_mean:.2f} rest_mean={rest_mean:.2f} "
                      f"ratio={mean_ratio:.2f} boundary={boundary_gap_pc:+.2f} noise_floor={noise_floor:.2f}")

            if (
                sus_mean > noise_floor and
                (mean_ratio > 1.25 or boundary_gap_pc > 1.0)
            ):
                perclass_reject = {candidate_tags_pc[i] for i in suspect_idx}
                for tid in perclass_reject:
                    if tid in stage1_pass:
                        stage1_fail[tid] = stage1_pass.pop(tid)
                        stage1_fail_reason[tid] = "PerClass"
                if perclass_reject:
                    print(f"      [Stage 1P PerClass] Rejected {len(perclass_reject)} clients "
                          f"(sus_mean={sus_mean:.2f}, ratio={mean_ratio:.2f}, gap={boundary_gap_pc:+.2f})")
                survivor_ratio = len(stage1_pass) / max(len(tag_ids), 1)

        if len(stage1_pass) >= 3:
            anchor = float(np.median(np.array(list(stage1_pass.values()), dtype=float)))
        else:
            pos_all = vals[vals > direction_eps]
            if len(pos_all) >= 3:
                anchor = float(np.median(pos_all))
            elif len(pos_all) > 0:
                anchor = float(np.mean(pos_all))
            else:
                anchor = self.prev_anchor

        anchor_floor = max(1e-3, 0.25 * self.prev_anchor)
        anchor = max(anchor, anchor_floor)
        self.prev_anchor = anchor

        # Round threshold scaled from MNIST's 10 -> 3 for Amazon's 60-round horizon.
        if round_num < 3:
            min_ratio, max_ratio = 0.10, 10.0
        elif survivor_ratio < 0.5:
            min_ratio, max_ratio = 0.02, 25.0
        elif survivor_ratio < 0.7:
            min_ratio, max_ratio = 0.05, 15.0
        else:
            min_ratio, max_ratio = 0.30, 3.0

        ref_group_scores = {}
        for tid, score in stage1_pass.items():
            ref_key = best_ref_map[tid].split('+', 1)[0]
            ref_group_scores.setdefault(ref_key, []).append(score)

        ref_group_bounds = {}
        ref_anchor_parts = []
        for ref_key, ref_scores in ref_group_scores.items():
            ref_arr = np.array(ref_scores, dtype=float)

            # Non-IID targeted: isolated groups (len==1) fall back to the global anchor
            # rather than the score itself, because single-sample groups carry
            # basically no information in pathological non-IID.
            if len(ref_arr) >= 3:
                ref_anchor = float(np.median(ref_arr))
            elif len(ref_arr) == 2:
                ref_anchor = float(np.mean(ref_arr))
            else:
                ref_anchor = anchor

            prev_group_anchor = self.prev_anchor_by_ref.get(ref_key, anchor)
            group_floor = max(1e-3, 0.10 * prev_group_anchor, 0.05 * anchor)
            ref_anchor = max(ref_anchor, group_floor)
            self.prev_anchor_by_ref[ref_key] = ref_anchor

            ref_spread = float(np.median(np.abs(ref_arr - np.median(ref_arr)))) if len(ref_arr) >= 2 else 0.0

            if len(ref_arr) == 1:
                group_min_ratio = min(min_ratio, 0.04)
                group_max_ratio = max(max_ratio, 20.0)
            elif len(ref_arr) <= 4:
                group_min_ratio = min(min_ratio, 0.10)
                group_max_ratio = max(max_ratio, 10.0)
            else:
                group_min_ratio = min_ratio
                group_max_ratio = max_ratio

            min_acc = ref_anchor * group_min_ratio
            max_acc = ref_anchor * group_max_ratio
            tiny_anchor = ref_anchor < max(0.02, 0.15 * anchor)
            if tiny_anchor:
                min_acc = min(min_acc, max(direction_eps, 0.0))
                max_acc = max(max_acc, float(np.max(ref_arr)) + max(3.0 * ref_spread, 0.01))

            ref_group_bounds[ref_key] = {
                'anchor': ref_anchor,
                'min_ratio': group_min_ratio,
                'max_ratio': group_max_ratio,
                'min_acc': min_acc,
                'max_acc': max_acc,
                'count': len(ref_arr),
            }
            ref_anchor_parts.append(f"{ref_key}:{ref_anchor:.4f}")

        default_bounds = {
            'anchor': anchor,
            'min_ratio': min_ratio,
            'max_ratio': max_ratio,
            'min_acc': anchor * min_ratio,
            'max_acc': anchor * max_ratio,
            'count': len(stage1_pass),
        }

        registry = self.key_authority.get_registry_snapshot()
        accepted = set()
        # Verbose every 5 rounds for Amazon's 60-round horizon (was every 10 on MNIST).
        verbose_round = ((round_num + 1) % 5 == 0)

        if verbose_round:
            ref_mix = ', '.join(f"{name}:{count}" for name, count in sorted(ref_usage.items()))
            ref_anchor_summary = ', '.join(ref_anchor_parts) if ref_anchor_parts else 'none'
            print(f"\n[FIP] Round {round_num + 1} | median={raw_median:.6f} | anchor={anchor:.6f}")
            print(f"      dir_eps={direction_eps:.6f} | global_bounds=({default_bounds['min_acc']:.6f}, {default_bounds['max_acc']:.6f})")
            print(f"      ratios=({min_ratio:.2f}x, {max_ratio:.2f}x) | collapse_streak={self.collapse_streak}")
            print(f"      stage1_pass={len(stage1_pass)} | stage1_fail={len(stage1_fail)} | ref_mix={ref_mix}")
            print(f"      ref_anchors={ref_anchor_summary}")
            if targeted_margin_debug is not None and targeted_margin_reject:
                _gap, _thr, _delta = targeted_margin_debug
                print(
                    f"      [Stage 1T] Targeted source-margin detection: rejected "
                    f"{len(targeted_margin_reject)} clients "
                    f"(gap={_gap:.6f}, thresh={_thr:.6f}, delta={_delta:.6f})"
                )
            if safe_rescue_tags:
                print(f"      [Stage 1R] Honest-safe rescue: restored {len(safe_rescue_tags)} clients")
            print(f"  {'Client':>8}  {'TAG':>8}  {'Type':>6}  {'Ref':>10}  {'Score':>10}  Result")
            print(f"  {'-' * 82}")

        for tid, score in scores_map.items():
            client_id = registry.get(tid, "Unknown")
            client_type = "MAL" if tid in mal_tags else "HON"
            best_ref_name = best_ref_map[tid]

            if tid in stage1_fail:
                if verbose_round:
                    fail_reason = stage1_fail_reason.get(tid, "Direction")
                    print(f"  {client_id:>8}  {tid:>8}  {client_type:>6}  {best_ref_name:>10}  {score:>10.4f}  REJECT (Stage 1: {fail_reason})")
                continue

            valid = True
            failed_stage = None
            trusted_safe = (
                (tid in guaranteed_honest_tags) or
                (targeted_margin_active and tid in safe_source_tags) or
                (tid in safe_rescue_tags)
            )

            ref_key = best_ref_name.split('+', 1)[0]
            bounds = ref_group_bounds.get(ref_key, default_bounds)
            min_acc = bounds['min_acc']
            max_acc = bounds['max_acc']

            if score < min_acc:
                if not (trusted_safe and score > (direction_eps - safe_slack)):
                    valid = False
                    failed_stage = f"Stage 2 (Too Small: {score:.4f} < {min_acc:.4f})"
            elif score > max_acc:
                valid = False
                failed_stage = f"Stage 2 (Too Large: {score:.4f} > {max_acc:.4f})"

            history_key = self._history_key(tid, best_ref_name)
            if valid:
                ok, reason = self._check_consistency(history_key, score, round_num)
                if not ok:
                    valid = False
                    failed_stage = f"Stage 3 (History: {reason})"

            if valid:
                if verbose_round:
                    print(f"  {client_id:>8}  {tid:>8}  {client_type:>6}  {best_ref_name:>10}  {score:>10.4f}  ACCEPT")
                accepted.add(tid)
                if history_key not in self.client_history:
                    self.client_history[history_key] = []
                self.client_history[history_key].append(score)
                if len(self.client_history[history_key]) > 10:
                    self.client_history[history_key].pop(0)
            else:
                if verbose_round:
                    print(f"  {client_id:>8}  {tid:>8}  {client_type:>6}  {best_ref_name:>10}  {score:>10.4f}  REJECT ({failed_stage})")

        acceptance_ratio = len(accepted) / len(tag_ids) if tag_ids else 1.0
        self.last_acceptance_ratio = acceptance_ratio
        self.empty_rounds = 0 if accepted else (self.empty_rounds + 1)
        self.last_reference_source = max(ref_usage, key=ref_usage.get)
        self.last_ref_usage = ref_usage
        self.last_stage1_pass_count = len(stage1_pass)
        self.last_stage1_fail_count = len(stage1_fail)
        self.last_round_size = len(tag_ids)
        positive_scores = np.array(list(stage1_pass.values()), dtype=float) if stage1_pass else np.array([], dtype=float)
        self.last_raw_median = raw_median
        self.last_positive_median = float(np.median(positive_scores)) if len(positive_scores) else direction_eps
        self.last_direction_eps = direction_eps

        if len(accepted) == 0 and verbose_round:
            print("  [Safety] No client accepted this round; aggregation skipped.")

        if verbose_round:
            print(f"  -> Accepted {len(accepted)}/{len(tag_ids)} clients")

        snapshot = self.update_table.copy()
        self.update_table = {}
        return {tid: {'count': 1, 'enc_data': snapshot[tid]} for tid in accepted}

    def feedback(self, global_grad):
        if global_grad is None:
            return

        survivor_ratio = self.last_stage1_pass_count / max(self.last_round_size, 1)
        min_survivors = max(4, int(np.ceil(0.4 * self.last_round_size)))
        healthy_round = (
            self.last_round_size > 0 and
            self.last_stage1_pass_count >= min_survivors and
            self.last_acceptance_ratio >= 0.35 and
            self.last_positive_median > max(self.last_direction_eps, 0.0)
        )

        if healthy_round:
            momentum = self.alpha_ref if self.last_acceptance_ratio > 0.7 else (0.90 if self.last_acceptance_ratio >= 0.45 else 0.95)
            new_ref = (momentum * self.ref_directions[0]) + ((1 - momentum) * global_grad)
            new_ref = self._normalize_reference(new_ref)
            self.ref_directions[0] = new_ref
            self.last_stable_ref = new_ref.copy()
            return

        recovery_target = self.last_stable_ref if self.last_stable_ref is not None else self.bootstrap_ref
        if self.collapse_streak >= 3:
            recovery_target = self.bootstrap_ref
        recovery_blend = 0.95 if self.collapse_streak < 3 else 0.99
        recovered_ref = (recovery_blend * recovery_target) + ((1.0 - recovery_blend) * self.ref_directions[0])
        self.ref_directions[0] = self._normalize_reference(recovered_ref)


# ============================================================
# CLIENT MANAGER
# ============================================================
class ClientManager:
    def __init__(self, auth, dm, dim):
        self.auth = auth
        self.dm = dm
        self.dim = dim
        self.clients_state_W = {}
        self.clients_state_b = {}
        self.client_configs = []

    def init_clients(self, n_clients, num_malicious):
        self.client_configs = []
        all_indices = list(range(n_clients))
        malicious_indices = np.random.choice(all_indices, num_malicious, replace=False)

        # Pull class-1 samples from the priv pool so malicious clients without native
        # class-1 shards still have material for the 1->7 flip (otherwise the attack
        # is a no-op in pathological non-IID).
        class_1_mask = self.dm.y_priv == 1
        X_class_1 = self.dm.X_priv[class_1_mask]
        y_class_1 = self.dm.y_priv[class_1_mask]

        for i in range(n_clients):
            is_mal = (i in malicious_indices)
            c_id = f"C{i}"
            X, y = self.dm.get_client_data(i)

            if is_mal and len(X_class_1) > 0:
                half = max(1, len(y) // 2)
                idx = np.random.choice(len(X_class_1), min(half, len(X_class_1)), replace=False)
                X_mod = np.copy(X)
                y_mod = np.copy(y)
                X_mod[:len(idx)] = X_class_1[idx]
                y_mod[:len(idx)] = y_class_1[idx]
                X = X_mod
                y = y_mod

            has_source_class = bool(np.any(y == 1))

            self.client_configs.append({
                'id': c_id,
                'X': X,
                'y': y,
                'mal': is_mal,
                'source_present': has_source_class,
                'reg_tag': self.auth.register_user(c_id)
            })
            self.clients_state_W[c_id] = np.zeros((self.dim, NUM_CLASSES))
            self.clients_state_b[c_id] = np.zeros(NUM_CLASSES)

        mal_ids = [f"C{i}" for i in malicious_indices]
        targeted_ids = sorted([cfg['id'] for cfg in self.client_configs if cfg['source_present']])
        print(f"  Malicious clients: {sorted(mal_ids)}")
        print(f"  Targeted-eligible clients (has class 1): {targeted_ids}")

    def run_training_round(self, global_W, global_b):
        jobs = []
        pub, _ = self.auth.distribute_keys()
        for cfg in self.client_configs:
            self.clients_state_W[cfg['id']] = copy.deepcopy(global_W)
            self.clients_state_b[cfg['id']] = copy.deepcopy(global_b)
            args = (
                cfg['id'], cfg['X'], cfg['y'],
                self.clients_state_W[cfg['id']], self.clients_state_b[cfg['id']],
                cfg['mal'], NUM_CLASSES, self.dim, pub, FIXED_POINT_SCALE,
                LEARNING_RATE, BATCH_SIZE, LOCAL_EPOCHS
            )
            jobs.append(args)

        with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
            results = list(executor.map(_train_client_worker, jobs))
        return results


# ============================================================
# SERVER
# ============================================================
class Server:
    def __init__(self, input_dim, num_classes=NUM_CLASSES):
        self.W = np.zeros((input_dim, num_classes))
        self.b = np.zeros(num_classes)
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.fp = FixedPoint()

    def aggregate(self, packet, auth_priv):
        if not packet:
            return None
        first = list(packet.keys())[0]
        agg_enc = packet[first]['enc_data'][:]
        count = 1
        keys = list(packet.keys())
        for i in range(1, len(keys)):
            data = packet[keys[i]]['enc_data']
            for j in range(len(agg_enc)):
                agg_enc[j] = agg_enc[j] + data[j]
            count += 1

        flat_agg = self.fp.dec_vec(auth_priv, agg_enc)
        avg_update = flat_agg / count
        w_size = self.input_dim * self.num_classes
        update_W = avg_update[:w_size].reshape(self.input_dim, self.num_classes)
        update_b = avg_update[w_size:]
        self.W -= update_W
        self.b -= update_b
        return avg_update

    def evaluate_detailed(self, X_test, y_test):
        scores = np.dot(X_test, self.W) + self.b
        preds = np.argmax(scores, axis=1)
        acc = accuracy_score(y_test, preds)
        t_mask = y_test == 1
        t_acc = accuracy_score(y_test[t_mask], preds[t_mask]) if any(t_mask) else 0.0
        o_mask = (y_test != 1) & (y_test != 7)
        o_acc = accuracy_score(y_test[o_mask], preds[o_mask]) if any(o_mask) else 0.0
        return t_acc, o_acc, acc


def generate_accuracy_metric_plots(history, attack_rates, n_rounds):
    styles = ['-', '--', '-.', ':']
    rounds = np.arange(1, n_rounds + 1)
    plot_specs = [
        ('acc', 'Overall Accuracy Over Rounds (Amazon Non-IID Targeted)', 'Accuracy', 'fip_amazon_noniid_targeted_overall_accuracy.png'),
        ('o_acc', 'Other-Class Accuracy Over Rounds (Amazon Non-IID Targeted)', 'Other-Class Accuracy', 'fip_amazon_noniid_targeted_other_accuracy.png'),
        ('t_acc', 'Target Accuracy Over Rounds (Amazon Non-IID Targeted)', 'Target Accuracy', 'fip_amazon_noniid_targeted_target_accuracy.png'),
    ]

    for metric_key, title, ylabel, filename in plot_specs:
        plt.figure(figsize=(10, 6))
        for i, ar in enumerate(attack_rates):
            mean_metric = np.mean(history[ar][metric_key], axis=0)
            plt.plot(rounds, mean_metric, linestyle=styles[i % len(styles)], linewidth=2, label=f"AR = {ar * 100:.0f}%")
        plt.title(title)
        plt.xlabel('Communication Round')
        plt.ylabel(ylabel)
        plt.ylim([-0.05, 1.05])
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(filename, dpi=300)
        plt.show()
        plt.close()


# ============================================================
# MAIN
# ============================================================
if __name__ == '__main__':
    multiprocessing.set_start_method('spawn', force=True)

    history = {
        ar: {
            'acc': np.zeros((len(SEEDS), N_ROUNDS)),
            't_acc': np.zeros((len(SEEDS), N_ROUNDS)),
            'o_acc': np.zeros((len(SEEDS), N_ROUNDS)),
            'tpr': np.zeros((len(SEEDS), N_ROUNDS)),
            'fpr': np.zeros((len(SEEDS), N_ROUNDS))
        }
        for ar in ATTACK_RATES
    }
    roc_data = {ar: {'y_true': [], 'y_score': []} for ar in ATTACK_RATES}
    all_round_results = []

    np.random.seed(42)
    dm = DataManager(N_CLIENTS)
    dim = dm.get_input_shape()

    total_steps = len(SEEDS) * len(ATTACK_RATES) * N_ROUNDS
    global_pbar = tqdm(total=total_steps, desc="Overall", ncols=80, disable=not USE_TQDM)

    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'=' * 60}")
        print(f" SEED: {seed}  ({seed_idx + 1}/{len(SEEDS)})")
        print(f"{'=' * 60}")

        np.random.seed(seed)

        for ar_idx, attack_rate in enumerate(ATTACK_RATES):
            num_malicious = int(N_CLIENTS * attack_rate)
            print(f"\n{'=' * 60}")
            print(f" EXP: Attack Rate={attack_rate * 100:.0f}%  ({num_malicious} malicious)  [{ar_idx + 1}/{len(ATTACK_RATES)}]")
            print(f" Attack: Targeted Label Flip 1->7 (Non-IID)")
            print(f"{'=' * 60}")

            auth = TrustedKeyAuthority()
            verifier = FIPVerifier(auth, dm, dim, NUM_CLASSES, attack_rate=attack_rate)
            server = Server(dim, NUM_CLASSES)
            client_mgr = ClientManager(auth, dm, dim)
            client_mgr.init_clients(N_CLIENTS, num_malicious)

            verifier.set_targeted_source_tags({
                c['reg_tag'] for c in client_mgr.client_configs if c['source_present']
            })

            X_test, y_test = dm.get_test_data()

            mal_tags = {c['reg_tag'] for c in client_mgr.client_configs if c['mal']}
            honest_tags = {c['reg_tag'] for c in client_mgr.client_configs if not c['mal']}
            actual_mal_ids = sorted([auth.registry[t] for t in mal_tags])
            print(f"\n  [Ground Truth] Malicious: {actual_mal_ids}")

            for r in range(N_ROUNDS):
                start_time = time.time()

                results = client_mgr.run_training_round(server.W, server.b)
                for res in results:
                    c_id, enc_grad, _ = res
                    tag = [c['reg_tag'] for c in client_mgr.client_configs if c['id'] == c_id][0]
                    verifier.receive_update(tag, enc_grad)

                packet = verifier.perform_fip_audit(round_num=r, mal_tags=mal_tags)

                if packet:
                    grad = server.aggregate(packet, auth.priv_key)
                    if grad is not None:
                        verifier.feedback(grad)

                t_acc, o_acc, acc = server.evaluate_detailed(X_test, y_test)
                history[attack_rate]['acc'][seed_idx, r] = acc
                history[attack_rate]['t_acc'][seed_idx, r] = t_acc
                history[attack_rate]['o_acc'][seed_idx, r] = o_acc

                accepted_tags = set(packet.keys()) if packet else set()
                rejected_tags = mal_tags.union(honest_tags) - accepted_tags

                TP = len(rejected_tags & mal_tags)
                FP = len(rejected_tags & honest_tags)
                tpr = TP / len(mal_tags) if len(mal_tags) > 0 else 1.0
                fpr = FP / len(honest_tags) if len(honest_tags) > 0 else 0.0

                history[attack_rate]['tpr'][seed_idx, r] = tpr
                history[attack_rate]['fpr'][seed_idx, r] = fpr

                TN = len(accepted_tags & honest_tags)
                all_round_results.append({
                    'Seed': seed,
                    'Attack_Rate': attack_rate,
                    'Round': r + 1,
                    'Acc': acc,
                    'Target_Acc': t_acc,
                    'Other_Acc': o_acc,
                    'TPR': tpr,
                    'FPR': fpr,
                    'Actual_Malicious_Detected': f"{TP}/{len(mal_tags)}",
                    'Actual_Honest_Detected': f"{TN}/{len(honest_tags)}",
                })

                dt = time.time() - start_time

                if (r + 1) % 5 == 0:
                    detected_ids = sorted([auth.registry[t] for t in rejected_tags])
                    accepted_ids = sorted([auth.registry[t] for t in accepted_tags])
                    true_positive_ids = sorted([auth.registry[t] for t in (rejected_tags & mal_tags)])
                    false_positive_ids = sorted([auth.registry[t] for t in (rejected_tags & honest_tags)])
                    print(
                        f"\n  Round {r + 1:>3}: ACC={acc:.4f} | t_acc={t_acc:.4f} | "
                        f"o_acc={o_acc:.4f} | TPR={tpr:.2f} | FPR={fpr:.2f}"
                    )
                    print(f"    Actual Malicious   : {actual_mal_ids}")
                    print(f"    Detected Malicious : {detected_ids}")
                    print(f"    True Positives     : {true_positive_ids}")
                    print(f"    False Positives    : {false_positive_ids}")
                    print(f"    Accepted Clients   : {accepted_ids}")
                    print(f"    Rejected Clients   : {detected_ids}")
                    print(f"    Round Time         : {dt:.2f}s")

                if (r + 1) % 20 == 0:
                    print(f"  [Time] Round {r + 1} took {dt:.2f}s")
                global_pbar.update(1)
                global_pbar.set_postfix({
                    'AR': f"{int(attack_rate*100)}%",
                    'r': f"{r+1}/{N_ROUNDS}",
                    'acc': f"{acc:.3f}",
                    'tpr': f"{tpr:.2f}",
                })
                if not USE_TQDM:
                    print(f"[AR={int(attack_rate*100)}%] Round {r+1}/{N_ROUNDS} acc={acc:.3f} tpr={tpr:.2f} fpr={fpr:.2f} dt={dt:.0f}s", flush=True)

            roc_data[attack_rate]['y_true'].extend(verifier.roc_labels)
            roc_data[attack_rate]['y_score'].extend(verifier.roc_scores)

    global_pbar.close()
    df_all_rounds = pd.DataFrame(all_round_results)
    df_all_rounds.to_csv('final_fip_amazon_noniid_targeted_all_round_results.csv', index=False)
    print(f"[CSV] Saved {len(df_all_rounds)} rows to 'final_fip_amazon_noniid_targeted_all_round_results.csv'")

    generate_accuracy_metric_plots(history, ATTACK_RATES, N_ROUNDS)

    print("\n" + "=" * 60)
    print(" FINAL RESULTS ACROSS ALL SEEDS")
    print("=" * 60)
    for ar in ATTACK_RATES:
        final_accs = history[ar]['acc'][:, -1]
        final_t_accs = history[ar]['t_acc'][:, -1]
        print(f"\nAttack Rate {ar * 100:.0f}%:")
        print(f"  Overall Accuracy : {np.mean(final_accs):.4f} +/- {np.std(final_accs):.4f}")
        print(f"  Target Accuracy  : {np.mean(final_t_accs):.4f} +/- {np.std(final_t_accs):.4f}")
